# 02 — Normalization & Feature Selection

This notebook covers:
- Total-count normalization
- Log1p transformation
- Highly Variable Gene (HVG) selection
- Regressing out confounders
- Feature scaling


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import scanpy as sc
import matplotlib.pyplot as plt
import config as cfg
from utils import load_adata, save_adata, ensure_dirs, plot_hvg

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor='white')
%matplotlib inline
print('Setup complete ✓')

## 1. Load Preprocessed Data

In [ ]:
adata = load_adata(cfg.ADATA_PROCESSED_PATH)
print(adata)

## 2. Save Raw Counts

In [ ]:
# Save filtered raw counts to adata.raw for later use (e.g. DE testing)
adata.raw = adata
print('Raw counts saved to adata.raw')

## 3. Normalize Total Counts

In [ ]:
sc.pp.normalize_total(adata, target_sum=cfg.NORMALIZATION['target_sum'])
print(f'Normalized to {cfg.NORMALIZATION["target_sum"]:.0f} counts/cell')

## 4. Log1p Transform

In [ ]:
sc.pp.log1p(adata)
print('Log1p transformation applied')

## 5. Highly Variable Genes

In [ ]:
sc.pp.highly_variable_genes(
    adata,
    min_mean=cfg.NORMALIZATION['hvg_min_mean'],
    max_mean=cfg.NORMALIZATION['hvg_max_mean'],
    min_disp=cfg.NORMALIZATION['hvg_min_disp'],
    flavor=cfg.NORMALIZATION['hvg_flavor'],
)
print(f"HVGs: {adata.var['highly_variable'].sum()} / {adata.n_vars} genes")
sc.pl.highly_variable_genes(adata)

## 6. Subset to HVGs

In [ ]:
adata = adata[:, adata.var['highly_variable']].copy()
print(f'After HVG subset: {adata.n_obs} cells × {adata.n_vars} genes')

## 7. Regress Out Confounders

In [ ]:
sc.pp.regress_out(adata, ['total_counts', 'pct_counts_mt'])
print('Confounders regressed out')

## 8. Scale Features

In [ ]:
sc.pp.scale(adata, max_value=10)
print('Data scaled to unit variance (clipped at 10)')

## 9. Save

In [ ]:
save_adata(adata, cfg.ADATA_NORMALIZED_PATH)
print('Done ✓')